In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
import os
from collections import Counter
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv


In [3]:
# ==========================================
# 📂 LOAD ALL CSV FILES (FIXED VERSION)
# ==========================================
import os
import pandas as pd

path = "/kaggle/input/datasets/nvcidd/cc-project"

files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".csv")]

df_list = []

for file in files:
    print("Loading:", file)
    try:
        temp = pd.read_csv(
            file,
            encoding='latin1',        # FIX Unicode error
            low_memory=False,
            on_bad_lines='skip'       # skip corrupted rows
        )
        df_list.append(temp)
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

# Merge all successfully loaded files
df = pd.concat(df_list, ignore_index=True)

print("✅ Combined Dataset Shape:", df.shape)

Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv
✅ Combined Dataset Shape: (3119345, 85)


In [4]:
# Remove infinite + NaN
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

print("After Cleaning:", df.shape)

After Cleaning: (2827876, 85)


In [5]:
# 🔥 CLEAN COLUMN NAMES (VERY IMPORTANT)
df.columns = df.columns.str.strip()

# Now this will work
X = df.drop(columns=["Label"])
y = df["Label"]

# Save feature order
feature_columns = X.columns.tolist()

In [6]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)

Classes: ['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack \x96 Brute Force'
 'Web Attack \x96 Sql Injection' 'Web Attack \x96 XSS']


In [7]:
df["Label_encoded"] = y_encoded

balanced_list = []

for label in df["Label"].unique():
    temp = df[df["Label"] == label]

    # cap max samples per class
    if len(temp) > 50000:
        temp = temp.sample(50000, random_state=42)

    balanced_list.append(temp)

df_balanced = pd.concat(balanced_list)

print("Balanced Shape:", df_balanced.shape)
print(df_balanced["Label"].value_counts())

Balanced Shape: (239603, 86)
Label
BENIGN                        50000
PortScan                      50000
DDoS                          50000
DoS Hulk                      50000
DoS GoldenEye                 10293
FTP-Patator                    7935
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Bot                            1956
Web Attack  Brute Force       1507
Web Attack  XSS                652
Infiltration                     36
Web Attack  Sql Injection       21
Heartbleed                       11
Name: count, dtype: int64


In [8]:
X = df_balanced.drop(columns=["Label", "Label_encoded"])
y = df_balanced["Label_encoded"]

# Convert to numeric
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
from sklearn.utils import resample

df_full = X.copy()
df_full["Label"] = y

df_benign = df_full[df_full["Label"] == 0]
df_attack = df_full[df_full["Label"] != 0]

# Determine minority size
min_size = min(len(df_benign), len(df_attack))

# Downsample both to same size
df_benign_bal = resample(df_benign, replace=False, n_samples=min_size, random_state=42)
df_attack_bal = resample(df_attack, replace=False, n_samples=min_size, random_state=42)

# Combine
df_balanced = pd.concat([df_benign_bal, df_attack_bal])

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42)

# Split
X = df_balanced.drop(columns=["Label"])
y = df_balanced["Label"]

In [12]:
from sklearn.model_selection import train_test_split

X = df_balanced.drop(columns=["Label"])
y = df_balanced["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=2,
    eval_metric='mlogloss',
    use_label_encoder=False
)

xgb_model.fit(X_train_scaled, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:20:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
from sklearn.metrics import classification_report

proba = xgb_model.predict_proba(X_test_scaled)

y_pred = []

for p in proba:
    max_idx = np.argmax(p)
    max_prob = p[max_idx]

    # If it's BENIGN but low confidence → treat as attack
    if max_idx == 0 and max_prob < 0.4:
        y_pred.append(1)  # force attack
    else:
        y_pred.append(max_idx)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9922
           1       0.99      0.97      0.98       109
           2       1.00      1.00      1.00      2621
           3       0.99      1.00      1.00       567
           4       1.00      1.00      1.00      2674
           5       1.00      0.99      0.99       317
           6       1.00      1.00      1.00       327
           7       1.00      1.00      1.00       425
           9       1.00      0.50      0.67         2
          10       1.00      1.00      1.00      2644
          11       1.00      1.00      1.00       288
          12       0.93      0.89      0.91        76
          13       0.00      0.00      0.00         2
          14       0.75      0.81      0.78        26

    accuracy                           1.00     20000
   macro avg       0.90      0.87      0.88     20000
weighted avg       1.00      1.00      1.00     20000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
from sklearn.ensemble import IsolationForest

# Extract benign samples (Label = 0)
benign_data = df_balanced[df_balanced["Label"] == 0]

X_benign = benign_data.drop(columns=["Label"])
X_benign = X_benign.apply(pd.to_numeric, errors='coerce').fillna(0)

X_benign_scaled = scaler.transform(X_benign)

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.10,
    random_state=42
)

iso_model.fit(X_benign_scaled)

IsolationForest(contamination=0.1, random_state=42)

In [17]:
scores = iso_model.decision_function(X_test_scaled)

print("Min score:", scores.min())
print("Max score:", scores.max())

Min score: -0.25610837685172816
Max score: 0.14788464967495646


In [18]:
scores = iso_model.decision_function(X_test_scaled)

threshold = -0.05  # tune this

pred = [1 if s < threshold else 0 for s in scores]

print("Anomalies:", sum(pred))

Anomalies: 4662


In [19]:
y_true = [0 if y == 0 else 1 for y in y_test]  # 0=benign, 1=attack

In [20]:
from sklearn.metrics import classification_report
print(classification_report(y_true, pred))

              precision    recall  f1-score   support

           0       0.61      0.94      0.74      9922
           1       0.87      0.40      0.55     10078

    accuracy                           0.67     20000
   macro avg       0.74      0.67      0.64     20000
weighted avg       0.74      0.67      0.64     20000



In [21]:
import pickle

pickle.dump(xgb_model, open("ids_xgboost_multiclass.pkl", "wb"))
pickle.dump(iso_model, open("isolation_forest.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))
pickle.dump(feature_columns, open("features.pkl", "wb"))

print("✅ ALL MODELS SAVED")

✅ ALL MODELS SAVED


In [22]:
from sklearn.metrics import classification_report

labels = sorted(set(y_test))

print(classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=le.inverse_transform(labels)
))

                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00      9922
                       Bot       0.99      0.97      0.98       109
                      DDoS       1.00      1.00      1.00      2621
             DoS GoldenEye       0.99      1.00      1.00       567
                  DoS Hulk       1.00      1.00      1.00      2674
          DoS Slowhttptest       1.00      0.99      0.99       317
             DoS slowloris       1.00      1.00      1.00       327
               FTP-Patator       1.00      1.00      1.00       425
              Infiltration       1.00      0.50      0.67         2
                  PortScan       1.00      1.00      1.00      2644
               SSH-Patator       1.00      1.00      1.00       288
  Web Attack  Brute Force       0.93      0.89      0.91        76
Web Attack  Sql Injection       0.00      0.00      0.00         2
          Web Attack  XSS       0.75      0.81

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
# Scale test data
X_test_scaled = scaler.transform(X_test)

# Predict
pred = iso_model.predict(X_test_scaled)

print("Anomalies detected:", sum(pred == -1))
print("Normal detected:", sum(pred == 1))

Anomalies detected: 5239
Normal detected: 14761


In [24]:
y_true = [0 if y == 0 else 1 for y in y_test]
y_pred_if = [1 if p == -1 else 0 for p in pred]

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred_if))

              precision    recall  f1-score   support

           0       0.61      0.90      0.73      9922
           1       0.82      0.42      0.56     10078

    accuracy                           0.66     20000
   macro avg       0.71      0.66      0.64     20000
weighted avg       0.71      0.66      0.64     20000



In [25]:
import pandas as pd

dfs = []

files = [
    "/kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv",   # BENIGN
    "/kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
]

# 🔥 DIFFERENT SAMPLE SIZES (SMART MIX)
sample_sizes = {
    "Monday": 20000,     # mostly benign
    "Tuesday": 25000,
    "Wednesday": 25000,
    "Thursday": 30000,
    "Friday": 30000
}

for f in files:
    print(f"Loading: {f}")
    
    try:
        df = pd.read_csv(f, encoding='latin1', low_memory=False)
    except:
        df = pd.read_csv(f, encoding='ISO-8859-1', low_memory=False)

    # 🔥 FIX COLUMN NAMES
    df.columns = df.columns.str.strip()

    # Decide sample size
    if "Monday" in f:
        n = sample_sizes["Monday"]
    elif "Tuesday" in f:
        n = sample_sizes["Tuesday"]
    elif "Wednesday" in f:
        n = sample_sizes["Wednesday"]
    elif "Thursday" in f:
        n = sample_sizes["Thursday"]
    else:
        n = sample_sizes["Friday"]

    if len(df) > n:
        df = df.sample(n=n, random_state=42)

    dfs.append(df)
    

# Combine + shuffle
df_final = pd.concat(dfs).sample(frac=1, random_state=42)

# Save
df_final.to_csv("final_realistic_dataset.csv", index=False)

print("✅ Final Dataset Shape:", df_final.shape)
print(df_final["Label"].value_counts())

Loading: /kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
✅ Final Dataset Shape: (160000, 85)
Label
BENIGN                      97472
DDoS                        17101
PortScan                    16663
DoS Hulk                     8253
FTP-Patator                   465
DoS GoldenEye                 344
SSH-Patator                   340
DoS slowloris                 203
DoS Slowhttptest              195
Web Attack  Brute Force       95
Web Attack  XSS               59
Heartbleed           

In [26]:
import pandas as pd

dfs = []

files = [
    "/kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "/kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
]

for f in files:
    print(f"Loading: {f}")
    
    try:
        df = pd.read_csv(f, encoding='latin1', low_memory=False)
    except:
        df = pd.read_csv(f, encoding='ISO-8859-1', low_memory=False)

    # Clean column names
    df.columns = df.columns.str.strip()

    # 🔥 BALANCED SAMPLING PER CLASS
    if "Label" in df.columns:
        df_small = df.groupby("Label").apply(
            lambda x: x.sample(n=min(len(x), 300), random_state=42)
        ).reset_index(drop=True)
    else:
        df_small = df.sample(n=min(len(df), 300), random_state=42)

    dfs.append(df_small)

# Combine + shuffle
df_final = pd.concat(dfs).sample(frac=1, random_state=42)

# 🔥 LIMIT FINAL SIZE (IMPORTANT)
df_final = df_final.head(2000)

# Save
df_final.to_csv("final_small_balanced.csv", index=False)

print("✅ Final Dataset Shape:", df_final.shape)
print(df_final["Label"].value_counts())

Loading: /kaggle/input/datasets/nvcidd/cc-project/Monday-WorkingHours.pcap_ISCX.csv


/tmp/ipykernel_55/2152656281.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby("Label").apply(


Loading: /kaggle/input/datasets/nvcidd/cc-project/Tuesday-WorkingHours.pcap_ISCX.csv


/tmp/ipykernel_55/2152656281.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby("Label").apply(


Loading: /kaggle/input/datasets/nvcidd/cc-project/Wednesday-workingHours.pcap_ISCX.csv


...

/tmp/ipykernel_55/2152656281.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby("Label").apply(


Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


/tmp/ipykernel_55/2152656281.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby("Label").apply(


Loading: /kaggle/input/datasets/nvcidd/cc-project/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
✅ Final Dataset Shape: (2000, 85)
Label
BENIGN                        757
DoS GoldenEye                 132
DoS Slowhttptest              129
SSH-Patator                   128
DDoS                          127
Web Attack  XSS              126
Web Attack  Brute Force      122
DoS slowloris                 121
FTP-Patator                   118
PortScan                      115
DoS Hulk                      112
Web Attack  Sql Injection      8
Heartbleed                      5
Name: count, dtype: int64


/tmp/ipykernel_55/2152656281.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby("Label").apply(


In [27]:
xgb_model.save_model("xgb_model.json")